In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/processed/energy.db")

raw = pd.read_csv("../data/raw/mec02_data_centres.csv")
raw.columns = ["statistic", "quarter", "category", "unit", "value"]
raw.to_sql("consumption", conn, if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS n_rows FROM consumption", conn)

,n_rows
0,132


In [2]:
query = """
SELECT
    CAST(SUBSTR(quarter, 1, 4) AS INTEGER) AS year,
    SUM(value) AS dc_calculated
FROM consumption
WHERE category = 'Data centres'
GROUP BY year
ORDER BY year
"""
dc_yearly = pd.read_sql(query, conn)
dc_yearly

,year,dc_calculated
0,2015,1240
1,2016,1483
2,2017,1762
3,2018,2183
4,2019,2489
5,2020,3031
6,2021,4013
7,2022,5274
8,2023,6339
9,2024,6974


In [3]:
published = pd.read_excel("../excel/dc_reconciliation.xlsx", sheet_name="cso_published")
published.to_sql("cso_published", conn, if_exists="replace", index=False)
published.head()

,Year,DC_published,Other_published,Total_published
0,2015,1240,23360,24600
1,2016,1482,23874,25356
2,2017,1762,23963,25725
3,2018,2182,24548,26730
4,2019,2490,24015,26505


In [4]:
query = """
WITH calc AS (
    SELECT
        CAST(SUBSTR(quarter, 1, 4) AS INTEGER) AS year,
        SUM(value) AS dc_calculated
    FROM consumption
    WHERE category = 'Data centres'
    GROUP BY year
)
SELECT
    calc.year,
    calc.dc_calculated,
    p.DC_published AS dc_published,
    calc.dc_calculated - p.DC_published AS difference,
    CASE WHEN ABS(calc.dc_calculated - p.DC_published) <= 1
         THEN 'PASS' ELSE 'CHECK' END AS result
FROM calc
JOIN cso_published AS p ON calc.year = p.Year
ORDER BY calc.year
"""
recon = pd.read_sql(query, conn)
recon

,year,dc_calculated,dc_published,difference,result
0,2015,1240,1240,0,PASS
1,2016,1483,1482,1,PASS
2,2017,1762,1762,0,PASS
3,2018,2183,2182,1,PASS
4,2019,2489,2490,-1,PASS
5,2020,3031,3030,1,PASS
6,2021,4013,4012,1,PASS
7,2022,5274,5273,1,PASS
8,2023,6339,6339,0,PASS
9,2024,6974,6973,1,PASS


In [5]:
conn.close()
